# EEG-Based Emotion Recognition 
## SEED Dataset — Signal Preprocessing + Deep Learning



**Dataset:** SEED (SJTU Emotion EEG Dataset) — 15 subjects, 3 emotion classes  
**Task:** Binary stress classification (Negative vs Neutral/Positive)  
**Approach:** Signal preprocessing → Topographic image generation → CNN + LSTM + Fusion (StressNet)

---
### Project Overview
This notebook implements a full EEG-based emotion recognition pipeline:
1. **Data loading & exploration** — SEED dataset (50,910 samples, 62 channels, 5 frequency bands)
2. **Signal preprocessing** — Band-pass filtering, azimuthal projection, topomap generation
3. **Feature extraction** — 64×64×5 topographic images + raw EEG sequences
4. **Deep learning models** — CNN-only, LSTM-only, StressNet (CNN+LSTM Fusion), EEGNet
5. **Evaluation** — Accuracy, ROC-AUC, PR-AUC, confusion matrices
6. **Final summary** — Visual comparison chart of all models

## Section 1: Imports & Dataset Loading
We load the pre-processed SEED dataset. The data is already segmented into 1-second epochs with differential entropy (DE) features extracted across 5 frequency bands (delta, theta, alpha, beta, gamma) for 62 EEG channels.

In [ ]:
# ================================================================
# SEED Dataset → StressNet (CNN + LSTM + Fusion)
# ================================================================

# 0. Imports
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, gc, multiprocessing
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.signal import butter, filtfilt
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, Input, Model
from tensorflow.keras.optimizers import Adam
import seaborn as sns

# ================================================================
# 1. Load SEED data
# ================================================================
def load_seed(folder='/kaggle/input/datasets/daviderusso7/seed-dataset'):
    """Load SEED dataset: returns (N, channels, time), labels, subject IDs"""
    eeg   = np.load(os.path.join(folder, 'DatasetCaricatoNoImage.npz'))['arr_0']
    lab   = np.load(os.path.join(folder, 'LabelsNoImage.npz'))['arr_0']
    subj  = np.load(os.path.join(folder, 'SubjectsNoImage.npz'))['arr_0']
    return eeg, lab, subj

eeg_all, labels_all, subject_idx = load_seed('/kaggle/input/datasets/daviderusso7/seed-dataset')
print("Raw EEG shape  :", eeg_all.shape)   # (n_trials, n_channels, time)
print("Labels shape   :", labels_all.shape)
print("Subjects       :", np.unique(subject_idx).size)
print(f"Time dimension : {eeg_all.shape[2]}, channels: {eeg_all.shape[1]}")

is_bandpassed = (eeg_all.shape[1] == 5)
print(f"Data is pre-processed frequency bands: {is_bandpassed}")

# ================================================================
# 2. Binary stress label
# ================================================================
# SEED labels: 0 = negative, 1 = neutral, 2 = positive
# Binary task: stress (negative=0) vs non-stress (neutral/positive → 1)
print("\nUnique labels:", np.unique(labels_all))
print("Raw label counts:", np.bincount(labels_all))

y = (labels_all != 0).astype(int)
print("Class distribution (binary):", np.bincount(y))
print("  0 = Stress (Negative emotion)")
print("  1 = Non-stress (Neutral or Positive emotion)")

## Section 2: Exploratory Data Analysis (EDA)
Before building models, we explore the dataset structure — label distribution, per-subject sample counts, and frequency band characteristics.

In [ ]:
# ================================================================
# EDA 1: Label distribution
# ================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Class balance ---
counts = np.bincount(y)
axes[0].bar(['Stress (0)', 'Non-Stress (1)'], counts,
            color=['#e05c3a', '#3a8bc4'], edgecolor='white', linewidth=0.5)
for i, c in enumerate(counts):
    axes[0].text(i, c + 200, str(c), ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Binary Class Distribution', fontsize=13)
axes[0].set_ylabel('Sample Count')

# --- Original 3-class distribution ---
orig_counts = np.bincount(labels_all)
axes[1].bar(['Negative', 'Neutral', 'Positive'], orig_counts,
            color=['#e05c3a', '#888780', '#3a8bc4'], edgecolor='white')
for i, c in enumerate(orig_counts):
    axes[1].text(i, c + 100, str(c), ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Original 3-Class Label Distribution', fontsize=13)
axes[1].set_ylabel('Sample Count')

# --- Per-subject sample count ---
subj_counts = [int((subject_idx == s).sum()) for s in range(15)]
axes[2].bar(range(15), subj_counts, color='#3a8bc4', edgecolor='white')
axes[2].axhline(np.mean(subj_counts), color='red', linestyle='--',
                linewidth=1.5, label=f'Mean = {np.mean(subj_counts):.0f}')
axes[2].set_xticks(range(15))
axes[2].set_xticklabels([f'S{i}' for i in range(15)], fontsize=9)
axes[2].set_title('Samples per Subject', fontsize=13)
axes[2].set_ylabel('Count'); axes[2].legend()

plt.suptitle('SEED Dataset — Exploratory Data Analysis', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ================================================================
# EDA 2: Frequency band power analysis
# ================================================================
# The dataset is organized as (N, 5, 62):
#   axis 1 = 5 frequency bands (delta, theta, alpha, beta, gamma)
#   axis 2 = 62 EEG channels
# We visualize mean band power per emotion class

band_names  = ['Delta(0.5-4Hz)', 'Theta(4-7Hz)', 'Alpha(8-12Hz)', 'Beta(13-30Hz)', 'Gamma(30-45Hz)']
emotion_labels = ['Negative', 'Neutral', 'Positive']
em_colors      = ['#e05c3a', '#888780', '#3a8bc4']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Mean band power per emotion ---
x = np.arange(5); w = 0.25
for ei, (en, ec) in enumerate(zip(emotion_labels, em_colors)):
    mask = labels_all == ei
    # eeg_all shape: (N, 5, 62) — mean over all channels and samples
    mean_power = eeg_all[mask].mean(axis=0).mean(axis=1)  # (5,)
    axes[0].bar(x + (ei-1)*w, mean_power, width=w, label=en,
                color=ec, edgecolor='white', linewidth=0.5)
axes[0].set_xticks(x); axes[0].set_xticklabels(band_names, fontsize=9)
axes[0].set_ylabel('Mean Feature Value')
axes[0].set_title('Mean EEG Power per Frequency Band & Emotion', fontsize=12)
axes[0].legend()

# --- Band power variance across subjects ---
var_per_band = []
for si in range(5):
    subj_means = [eeg_all[subject_idx == s, si, :].mean() for s in range(15)]
    var_per_band.append(np.std(subj_means))

axes[1].bar(band_names, var_per_band,
            color=['#5dade2','#58d68d','#f5b041','#ec7063','#a569bd'],
            edgecolor='white')
axes[1].set_ylabel('Std of Subject Means')
axes[1].set_title('Inter-Subject Variability per Frequency Band', fontsize=12)
axes[1].set_xticklabels(band_names, fontsize=9)

plt.suptitle('EEG Frequency Band Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ================================================================
# EDA 3: Channel-wise activity heatmap
# ================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ei, (en, ec) in enumerate(zip(emotion_labels, em_colors)):
    mask = labels_all == ei
    # Mean across all bands → (62,) — one value per channel
    ch_activity = eeg_all[mask].mean(axis=0).mean(axis=0)  # (62,)
    axes[ei].bar(range(62), ch_activity, color=ec, alpha=0.8, width=0.9)
    axes[ei].set_title(f'{en} — Mean Activity per Channel', fontsize=11)
    axes[ei].set_xlabel('Channel Index (0–61)')
    axes[ei].set_ylabel('Mean DE Feature Value')
    axes[ei].set_xlim(-1, 63)

plt.suptitle('Channel-wise EEG Activity by Emotion Class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ================================================================
# EDA 4: Correlation between frequency bands
# ================================================================
# Compute correlation matrix between band-averaged features
band_data = eeg_all.mean(axis=2)   # (N, 5) — avg over channels
band_df   = pd.DataFrame(band_data,
            columns=['Delta','Theta','Alpha','Beta','Gamma'])

plt.figure(figsize=(7, 5))
sns.heatmap(band_df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Pearson Correlation Between Frequency Bands', fontsize=13)
plt.tight_layout(); plt.show()
print("Band correlation matrix printed above.")

## Section 3: Signal Preprocessing — EEG Channel Positions & Topographic Mapping
EEG channels are physically placed on the scalp following the 10-20 international system. We project 3D electrode positions onto a 2D azimuthal plane to create topographic images that encode spatial brain activity patterns.

In [ ]:
# ================================================================
# 3. Channel positions (62-channel SEED montage)
# ================================================================
seed_62 = [
    'FP1','FPZ','FP2','AF3','AF4','F7','F5','F3','F1','FZ','F2','F4','F6','F8',
    'FT7','FC5','FC3','FC1','FCZ','FC2','FC4','FC6','FT8','T7','C5','C3','C1',
    'CZ','C2','C4','C6','T8','TP7','CP5','CP3','CP1','CPZ','CP2','CP4','CP6',
    'TP8','P7','P5','P3','P1','PZ','P2','P4','P6','P8','PO7','PO5','PO3',
    'POZ','PO4','PO6','PO8','CB1','O1','OZ','O2','CB2'
]

import mne
montage   = mne.channels.make_standard_montage('standard_1020')
pos       = montage.get_positions()['ch_pos']
pos_seed  = {ch.upper(): pos[ch.upper()] for ch in seed_62 if ch.upper() in pos}
missing   = [c for c in seed_62 if c.upper() not in pos_seed]
print("Missing channels (Fibonacci-placed):", missing)

def fibonacci_sphere(n):
    i = np.arange(n) + 0.5
    phi   = np.arccos(1 - 2*i/n)
    theta = np.pi * (1 + 5**0.5) * i
    return np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)

Xfb, Yfb, Zfb = fibonacci_sphere(len(missing))
k_fb = np.sqrt(2.0 / (1.0 + Zfb))
u_fb = (k_fb * Xfb).astype(float)
v_fb = (k_fb * Yfb).astype(float)

u_ch = np.zeros(62); v_ch = np.zeros(62)
for i, ch in enumerate(seed_62):
    if ch.upper() in pos_seed:
        xyz     = pos_seed[ch.upper()]
        k       = np.sqrt(2.0 / (1.0 + xyz[2]))
        u_ch[i] = k * xyz[0]
        v_ch[i] = k * xyz[1]
    else:
        idx     = missing.index(ch)
        u_ch[i] = u_fb[idx]
        v_ch[i] = v_fb[idx]

# Visualize electrode positions
plt.figure(figsize=(6, 6))
plt.scatter(u_ch, v_ch, c='steelblue', s=60, zorder=5)
for i, ch in enumerate(seed_62):
    plt.text(u_ch[i], v_ch[i]+0.01, ch, fontsize=5, ha='center', va='bottom')
plt.title('62-Channel SEED Electrode Positions (Azimuthal Projection)', fontsize=11)
plt.axis('equal'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"Electrode positions computed for {len(seed_62)} channels.")

## Section 4: Band-pass Filtering & Topographic Image Generation
For each EEG trial, we:
1. Apply a Butterworth band-pass filter for each of the 5 frequency bands
2. Compute mean absolute amplitude per channel per band
3. Interpolate channel values onto a 64×64 grid using cubic interpolation
4. Apply Gaussian smoothing to create smooth topographic maps

This yields a **64×64×5 multi-channel topographic image** per trial — spatially encoding brain activity across all frequency bands simultaneously.

In [ ]:
# ================================================================
# 4. Band-pass filtering + azimuthal topographic images (64×64×5)
# ================================================================
eeg_for_band = np.transpose(eeg_all, (0, 2, 1))    # (N, time=62, ch=5)
sf       = 200                                      # SEED sampling rate (Hz)
bands    = {'delta':(0.5,4),'theta':(4,7),'alpha':(8,12),'beta':(13,30),'gamma':(30,45)}
img_size = 64

def _butter(low, high, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return b, a

def _bandpass(eeg_3d, bname):
    lo, hi   = bands[bname]
    b, a     = _butter(lo, hi, sf)
    sig_len  = eeg_3d.shape[-1]
    padlen   = min(sig_len // 2 - 1, 27)
    if padlen < 1 or sig_len < 3:
        return eeg_3d.astype(np.float32)
    try:
        return filtfilt(b, a, eeg_3d, axis=-1, padlen=padlen).astype(np.float32)
    except ValueError:
        return eeg_3d.astype(np.float32)

u_grid = np.linspace(u_ch.min(), u_ch.max(), img_size)
v_grid = np.linspace(v_ch.min(), v_ch.max(), img_size)
UU, VV = np.meshgrid(u_grid, v_grid)
tree   = cKDTree(np.column_stack((u_ch, v_ch)))

def sample_to_topomap(vals, sigma=1.0):
    if np.any(np.isnan(vals)) or np.std(vals) < 1e-8:
        vals = np.nan_to_num(vals, nan=0.0)
    grid = griddata(np.column_stack((u_ch, v_ch)), vals,
                    (UU, VV), method='cubic')
    nan_mask = np.isnan(grid)
    if nan_mask.any():
        try:
            idx = tree.query(np.column_stack((UU.ravel()[nan_mask],
                                              VV.ravel()[nan_mask])))[1]
            grid.ravel()[nan_mask] = vals[idx]
        except:
            grid[nan_mask] = 0.0
    grid = gaussian_filter(grid, sigma=sigma)
    mn, mx = grid.min(), grid.max()
    grid = (grid-mn)/(mx-mn) if mx-mn > 1e-8 else np.zeros_like(grid)
    return np.clip(grid, 0, 1).astype(np.float32)

def gen_one_image(i, eeg_sample):
    out = np.zeros((img_size, img_size, len(bands)), dtype=np.float32)
    for b_idx, bname in enumerate(bands):
        filt         = _bandpass(eeg_sample[np.newaxis, :, :], bname)[0]
        vals         = np.mean(np.abs(filt), axis=1)
        out[..., b_idx] = sample_to_topomap(vals)
    return out

n_jobs = min(multiprocessing.cpu_count()-1, 8)
print("Generating 64×64×5 topographic images …")
imgs   = Parallel(n_jobs=n_jobs)(
    delayed(gen_one_image)(i, eeg_for_band[i])
    for i in tqdm(range(eeg_for_band.shape[0]))
)
images = np.stack(imgs)
print("Images shape:", images.shape)   # (50910, 64, 64, 5)
del imgs, eeg_for_band; gc.collect()

In [ ]:
# ================================================================
# Visualize sample topographic images (one per emotion class)
# ================================================================
band_labels = ['Delta','Theta','Alpha','Beta','Gamma']

fig, axes = plt.subplots(3, 5, figsize=(16, 9))
for ei, en in enumerate(emotion_labels):
    sample_idx = np.where(labels_all == ei)[0][42]   # pick a representative sample
    for bi in range(5):
        ax = axes[ei, bi]
        im = ax.imshow(images[sample_idx, :, :, bi], cmap='RdYlBu_r',
                       origin='lower', vmin=0, vmax=1)
        ax.axis('off')
        if ei == 0:
            ax.set_title(band_labels[bi], fontsize=11, fontweight='bold')
        if bi == 0:
            ax.set_ylabel(en, fontsize=11, rotation=90, labelpad=40)

plt.suptitle('Sample Topographic Maps (64×64) per Emotion × Frequency Band',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("Each row = one emotion class. Each column = one frequency band.")
print("Colour: red = high activity, blue = low activity.")

## Section 5: Data Splitting & Normalization
We create clean train/validation/test splits **before** fitting any normalization scaler — this prevents data leakage from the test set into training statistics.

- **Train:** 64% | **Validation:** 16% | **Test:** 20%  
- Scaler is fit **only on training data**, then applied to val and test.

In [ ]:
# ================================================================
# 5. Down-sample (if needed) + create splits
# ================================================================
down_factor = 8 if eeg_all.shape[2] > 1000 else 1
eeg_down    = eeg_all[:, :, ::down_factor]
print("EEG shape after downsampling:", eeg_down.shape)
print(f"Down-sample factor used: {down_factor}")

# Normalize images per-sample
for i in range(images.shape[0]):
    im = images[i]; mn, mx = im.min(), im.max()
    if mx-mn > 1e-8: images[i] = (im-mn)/(mx-mn)

# Deterministic stratified splits
idx = np.arange(len(y))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)
tr_idx, va_idx = train_test_split(tr_idx, test_size=0.2, stratify=y[tr_idx], random_state=42)

eeg_tr, eeg_va, eeg_te = eeg_down[tr_idx], eeg_down[va_idx], eeg_down[te_idx]
img_tr, img_va, img_te = images[tr_idx],   images[va_idx],   images[te_idx]
y_tr,   y_va,   y_te   = y[tr_idx],        y[va_idx],        y[te_idx]

# Fit scaler ONLY on train (no leakage)
scaler_eeg  = StandardScaler().fit(eeg_tr.reshape(-1, eeg_tr.shape[-1]))

def scale_split(X, sc):
    flat = X.reshape(-1, X.shape[-1])
    return sc.transform(flat).reshape(X.shape)

X_tr_e = scale_split(eeg_tr, scaler_eeg)
X_va_e = scale_split(eeg_va, scaler_eeg)
X_te_e = scale_split(eeg_te, scaler_eeg)

X_tr_i, X_va_i, X_te_i = img_tr, img_va, img_te

# Save splits
for name, arr in [('X_tr_i',X_tr_i),('X_va_i',X_va_i),('X_te_i',X_te_i),
                  ('X_tr_e',X_tr_e),('X_va_e',X_va_e),('X_te_e',X_te_e),
                  ('y_tr',y_tr),('y_va',y_va),('y_te',y_te)]:
    np.save(f'{name}.npy', arr.astype(np.float32 if 'y' not in name else np.int32))

print(f"\nSplit summary:")
print(f"  Train  : {len(y_tr):6d} samples  | class dist: {np.bincount(y_tr)}")
print(f"  Val    : {len(y_va):6d} samples  | class dist: {np.bincount(y_va)}")
print(f"  Test   : {len(y_te):6d} samples  | class dist: {np.bincount(y_te)}")
print(f"\nEEG shape  (train): {X_tr_e.shape}")
print(f"Image shape (train): {X_tr_i.shape}")

## Section 6: Load Splits & Verify
Reload the saved splits and confirm shapes are correct before model training.

In [ ]:
# ================================================================
# 6. Load cleaned splits
# ================================================================
import numpy as np, os

X_tr_i = np.load('X_tr_i.npy')
X_va_i = np.load('X_va_i.npy')
X_te_i = np.load('X_te_i.npy')

X_tr_e = np.load('X_tr_e.npy')
X_va_e = np.load('X_va_e.npy')
X_te_e = np.load('X_te_e.npy')

y_tr = np.load('y_tr.npy')
y_va = np.load('y_va.npy')
y_te = np.load('y_te.npy')

# EEG input to LSTM must be (N, time_steps, channels)
if X_tr_e.shape[1] > X_tr_e.shape[2]:
    X_tr_e = np.transpose(X_tr_e, (0,2,1))
    X_va_e = np.transpose(X_va_e, (0,2,1))
    X_te_e = np.transpose(X_te_e, (0,2,1))

print("EEG  shapes  — Train:", X_tr_e.shape, "Val:", X_va_e.shape, "Test:", X_te_e.shape)
print("Image shapes — Train:", X_tr_i.shape, "Val:", X_va_i.shape, "Test:", X_te_i.shape)
print("Label counts — Train:", np.bincount(y_tr),
      "Val:", np.bincount(y_va), "Test:", np.bincount(y_te))
assert len(np.unique(y_tr)) > 1 and len(np.unique(y_te)) > 1, "Labels degenerate!"
print("\nAll splits verified ✓")

## Section 7: Model Architecture Definitions

We implement four models:

1. **CNN branch** — Processes 64×64×5 topographic images. Three Conv2D blocks capture spatial patterns of brain activity across the scalp.
2. **LSTM branch** — Processes raw EEG sequences (time_steps × channels). Bidirectional LSTM layers capture temporal dynamics.
3. **StressNet (Fusion)** — Concatenates CNN and LSTM feature vectors, then passes through a fusion head. This multi-modal approach is our primary model.
4. **EEGNet** — A compact, EEG-specific architecture using depthwise separable convolutions, designed specifically for BCI tasks.

In [ ]:
# ================================================================
# 7. Model architecture definitions
# ================================================================
import tensorflow as tf
from tensorflow.keras import layers, Input, Model
from tensorflow.keras.optimizers import Adam

# ── CNN branch (processes topographic images) ─────────────────
def cnn_branch(input_shape=(64,64,5)):
    inp = Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(0.4)(x)

    feat = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(256, activation='relu')(feat)
    feat = layers.Dropout(0.5)(feat)
    return inp, feat

# ── LSTM branch (processes raw EEG time-series) ───────────────
def lstm_branch(time_steps, n_ch):
    inp = Input(shape=(time_steps, n_ch), name='eeg_input')
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)
    )(inp)
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=0.4, recurrent_dropout=0.4)
    )(x)
    x   = layers.Dropout(0.4)(x)
    feat = layers.Dense(128, activation='relu', name='lstm_feat')(x)
    return inp, feat

# ── StressNet: CNN + LSTM fusion ──────────────────────────────
def build_stressnet_fusion(img_shape, time_steps, n_ch, lr=1e-4):
    img_in, img_feat = cnn_branch(img_shape)
    eeg_in, eeg_feat = lstm_branch(time_steps, n_ch)

    x = layers.Concatenate(name='concat_feats')([img_feat, eeg_feat])
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = Model([img_in, eeg_in], out, name='StressNet_Fusion')
    model.compile(
        optimizer=Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='roc_auc'),
                 tf.keras.metrics.AUC(name='pr_auc', curve='PR')]
    )
    return model

# ── EEGNet (compact BCI-specific architecture) ────────────────
def build_eegnet(n_channels, n_samples, nb_classes=1, dropout_rate=0.5):
    inp = Input((n_samples, n_channels))
    x   = layers.Conv1D(8, 3, padding='same', activation='relu')(inp)
    x   = layers.BatchNormalization()(x)
    try:
        x = layers.DepthwiseConv1D(kernel_size=3, depth_multiplier=2, padding='same')(x)
    except:
        x = layers.SeparableConv1D(8, 3, padding='same')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('elu')(x)
    x   = layers.AveragePooling1D(2)(x)
    x   = layers.Dropout(dropout_rate)(x)
    x   = layers.SeparableConv1D(16, 3, padding='same', activation='elu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.AveragePooling1D(2)(x)
    x   = layers.Dropout(dropout_rate)(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(8, activation='elu')(x)
    out = layers.Dense(nb_classes, activation='sigmoid')(x)
    return Model(inp, out, name='EEGNet')

# ── Model summaries ───────────────────────────────────────────
cnn_demo = build_eegnet(X_tr_e.shape[2], X_tr_e.shape[1])
print(f"EEGNet parameters  : {cnn_demo.count_params():,}")
fusion_demo = build_stressnet_fusion((64,64,5), X_tr_e.shape[1], X_tr_e.shape[2])
print(f"StressNet parameters: {fusion_demo.count_params():,}")
del cnn_demo, fusion_demo

## Section 8: Train CNN-only and LSTM-only Models
We train individual CNN and LSTM models first as baselines, before combining them in the fusion model.

In [ ]:
# ================================================================
# 8. Train CNN-only model
# ================================================================
print("\n=== Training CNN-only ===")
cnn_in, cnn_feat = cnn_branch()
cnn_out = layers.Dense(1, activation='sigmoid')(
              layers.Dense(256, activation='relu')(cnn_feat))
cnn = Model(cnn_in, cnn_out)
cnn.compile(Adam(1e-4), 'binary_crossentropy',
            metrics=['accuracy',
                     tf.keras.metrics.AUC(),
                     tf.keras.metrics.AUC(curve='PR', name='pr_auc')])
cw_vals = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
cb_list = [callbacks.EarlyStopping(patience=8, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-7),
           callbacks.ModelCheckpoint('best_cnn.keras', save_best_only=True)]

cnn_history = cnn.fit(X_tr_i, y_tr, validation_data=(X_va_i, y_va),
    epochs=20, batch_size=32,
    class_weight=dict(enumerate(cw_vals)),
    callbacks=cb_list, verbose=1)

# ================================================================
# 8b. Train LSTM-only model
# ================================================================
print("\n=== Training LSTM-only ===")
lstm_in, lstm_feat = lstm_branch(X_tr_e.shape[1], X_tr_e.shape[2])
lstm_out = layers.Dense(1, activation='sigmoid')(
               layers.Dense(128, activation='relu')(lstm_feat))
lstm = Model(lstm_in, lstm_out)
lstm.compile(Adam(1e-4), 'binary_crossentropy',
             metrics=['accuracy',
                      tf.keras.metrics.AUC(),
                      tf.keras.metrics.AUC(curve='PR', name='pr_auc')])

rng        = np.random.RandomState(42)
X_tr_noisy = X_tr_e + rng.normal(0, 0.2, X_tr_e.shape)   # Noise augmentation on train only — val/test use clean data intentionally

lstm_history = lstm.fit(X_tr_noisy, y_tr,
    validation_data=(X_va_e, y_va),
    epochs=40, batch_size=32,
    class_weight=dict(enumerate(cw_vals)),
    callbacks=[callbacks.EarlyStopping(patience=8, restore_best_weights=True),
               callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-7),
               callbacks.ModelCheckpoint('best_lstm.keras', save_best_only=True)],
    verbose=1)

## Section 9: Train StressNet — CNN + LSTM Fusion
StressNet concatenates the spatial features from the CNN (topographic maps) with the temporal features from the LSTM (raw EEG sequence). The fusion head then learns to combine both modalities for the final classification.

In [ ]:
# ================================================================
# 9. Train StressNet (CNN + LSTM Fusion)
# ================================================================
print("\n=== Training StressNet (Fusion) ===")
from sklearn.utils.class_weight import compute_class_weight

fusion_model = build_stressnet_fusion(
    img_shape=(64,64,5),
    time_steps=X_tr_e.shape[1],
    n_ch=X_tr_e.shape[2]
)

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
class_weights = {0: float(cw[0]), 1: float(cw[1])}

history = fusion_model.fit(
    [X_tr_i, X_tr_e], y_tr,
    validation_data=([X_va_i, X_va_e], y_va),
    epochs=40, batch_size=32,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        callbacks.ModelCheckpoint('fusion.keras', save_best_only=True)
    ],
    verbose=1
)

## Section 10: Training Curves
Visualizing loss and accuracy curves helps diagnose overfitting and confirm convergence.

In [ ]:
# ================================================================
# 10. Training curves — Loss + Accuracy for all 3 models
# ================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

model_histories = [
    ('CNN-only',        cnn_history),
    ('LSTM-only',       lstm_history),
    ('StressNet Fusion',history),
]

for col, (name, hist) in enumerate(model_histories):
    # Loss
    axes[0, col].plot(hist.history['loss'],     label='Train Loss', color='#e05c3a')
    axes[0, col].plot(hist.history['val_loss'], label='Val Loss',   color='#3a8bc4', linestyle='--')
    axes[0, col].set_title(f'{name} — Loss',  fontsize=12)
    axes[0, col].set_xlabel('Epoch'); axes[0, col].set_ylabel('Loss')
    axes[0, col].legend(); axes[0, col].grid(True, alpha=0.3)

    # Accuracy
    axes[1, col].plot(hist.history['accuracy'],     label='Train Acc', color='#e05c3a')
    axes[1, col].plot(hist.history['val_accuracy'], label='Val Acc',   color='#3a8bc4', linestyle='--')
    axes[1, col].set_title(f'{name} — Accuracy', fontsize=12)
    axes[1, col].set_xlabel('Epoch'); axes[1, col].set_ylabel('Accuracy')
    axes[1, col].legend(); axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Training Curves — CNN, LSTM, StressNet', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Section 11: Train EEGNet
EEGNet is a compact, well-validated deep learning architecture specifically designed for EEG-based BCIs. It uses depthwise separable convolutions to efficiently learn spatial and temporal EEG features with far fewer parameters than standard CNNs.

In [ ]:
# ================================================================
# 11. Train EEGNet
# ================================================================
print("\n=== Training EEGNet ===")

def ensure_1d(y):
    y = np.asarray(y)
    if y.ndim > 1:
        return np.argmax(y, axis=1) if y.shape[1] > 1 else y.ravel()
    return y

eegnet = build_eegnet(X_tr_e.shape[2], X_tr_e.shape[1])
eegnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])

cw2       = compute_class_weight('balanced', classes=np.array([0,1]), y=ensure_1d(y_tr))
eeg_cw    = {0: float(cw2[0]), 1: float(cw2[1])}


eegnet_history = eegnet.fit(
    X_tr_e, y_tr,
    epochs=15, batch_size=32,
    validation_data=(X_va_e, y_va),
    class_weight=eeg_cw,
    callbacks=[
        callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        callbacks.ModelCheckpoint('best_eegnet.keras', save_best_only=True)
    ],
    verbose=1
)

# EEGNet training curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(eegnet_history.history['loss'],     label='Train', color='#e05c3a')
axes[0].plot(eegnet_history.history['val_loss'], label='Val',   color='#3a8bc4', linestyle='--')
axes[0].set_title('EEGNet — Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(eegnet_history.history['accuracy'],     label='Train', color='#e05c3a')
axes[1].plot(eegnet_history.history['val_accuracy'], label='Val',   color='#3a8bc4', linestyle='--')
axes[1].set_title('EEGNet — Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('EEGNet Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Section 12: Model Evaluation on Test Set
We evaluate all four models on the held-out test set. Metrics reported:
- **Accuracy** — overall correct predictions
- **ROC-AUC** — area under the ROC curve (discrimination ability)  
- **PR-AUC** — area under the Precision-Recall curve (important for imbalanced classes)

In [ ]:
# ================================================================
# 12. Evaluate all models on test set
# ================================================================
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score,
                              accuracy_score, f1_score, cohen_kappa_score)

def eval_model(mod, X, y_true, name, is_keras=True):
    if is_keras:
        prob = mod.predict(X, verbose=0).ravel()
    else:
        prob = mod.predict_proba(X)[:, 1]
    pred = (prob > 0.5).astype(int)
    acc  = accuracy_score(y_true, pred)
    roc  = roc_auc_score(y_true, prob)
    pr   = average_precision_score(y_true, prob)
    f1   = f1_score(y_true, pred)
    kap  = cohen_kappa_score(y_true, pred)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_true, pred,
          target_names=['Stress (0)', 'Non-Stress (1)'], digits=4))
    print(f"  Accuracy : {acc:.4f}  |  ROC-AUC : {roc:.4f}  |  PR-AUC : {pr:.4f}")
    print(f"  F1-Score : {f1:.4f}  |  Cohen κ  : {kap:.4f}")
    return {'name':name,'acc':acc,'roc':roc,'pr':pr,'f1':f1,'kappa':kap,
            'pred':pred,'prob':prob,'cm':confusion_matrix(y_true, pred)}

# Load best weights
cnn.load_weights('best_cnn.keras')
lstm.load_weights('best_lstm.keras')
fusion_model.load_weights('fusion.keras')

res_cnn    = eval_model(cnn,          X_te_i,          y_te, 'CNN-only')
res_lstm   = eval_model(lstm,         X_te_e,          y_te, 'LSTM-only')
res_fusion = eval_model(fusion_model, [X_te_i, X_te_e],y_te, 'StressNet (CNN+LSTM Fusion)')
res_eegnet = eval_model(eegnet,       X_te_e,          y_te, 'EEGNet')

all_results = [res_cnn, res_lstm, res_fusion, res_eegnet]

## Section 13: Confusion Matrices

In [ ]:
# ================================================================
# 13. Confusion matrices — all 4 models
# ================================================================
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
cmaps = ['Blues', 'Greens', 'Oranges', 'Purples']

for ax, res, cmap in zip(axes, all_results, cmaps):
    cm  = res['cm']
    pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    # Annotate with count + percentage
    annot = np.array([[f"{cm[i,j]}\n({pct[i,j]:.1f}%)"
                       for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=annot, fmt='', cmap=cmap, ax=ax,
                xticklabels=['Stress','Non-Stress'],
                yticklabels=['Stress','Non-Stress'],
                linewidths=0.5, cbar=False)
    ax.set_title(f"{res['name']}\nAcc: {res['acc']:.4f}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Section 14: Final Results Summary — Visual Comparison Chart
The chart below replicates the style from the project overview image, showing accuracy for all models including classical baselines (FFT+LDA, PCA+LDA, Naive Bayes with SGWT) as reference points from literature.

In [ ]:
# ================================================================
# 14. FINAL RESULTS CHART
# ================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

dl_names = [
    'FFT + LDA',
    'PCA + LDA',
    'Naive Bayes\n(SGWT)',
    'LSTM only',
    'EEGNet',
    'CNN only',
    'StressNet\n(CNN+LSTM Fusion)',
]
dl_accs = [
    61.23,
    77.28,
    78.20,
    res_lstm['acc']   * 100,
    res_eegnet['acc'] * 100,
    res_cnn['acc']    * 100,
    res_fusion['acc'] * 100,
]
colors = [
    '#5d6d7e', '#7f8c8d', '#27ae60',
    '#2980b9', '#2980b9', '#2980b9', '#1a5276'
]

# Sort ascending (barh renders bottom-to-top, so highest ends up at top)
combined = sorted(zip(dl_accs, dl_names, colors), key=lambda x: x[0])
dl_accs_sorted, dl_names_sorted, colors_sorted = zip(*combined)

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(dl_names_sorted, dl_accs_sorted, color=colors_sorted,
               edgecolor='white', linewidth=0.5, height=0.6)

for bar, acc in zip(bars, dl_accs_sorted):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{acc:.2f}%', va='center', ha='left',
            fontsize=11, fontweight='bold')

ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_xlim(50, 108)
ax.set_title('EEG Emotion Recognition — Model Accuracy Comparison\n(SEED Dataset)',
             fontsize=14, fontweight='bold')
ax.axvline(80, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax.axvline(90, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax.axvline(100, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax.grid(axis='x', alpha=0.3)

legend_patches = [
    mpatches.Patch(color='#5d6d7e', label='Classical Signal Processing'),
    mpatches.Patch(color='#27ae60', label='Classical ML (SGWT features)'),
    mpatches.Patch(color='#2980b9', label='Deep Learning'),
    mpatches.Patch(color='#1a5276', label='StressNet — Proposed Method'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('final_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: final_accuracy_comparison.png")

# ── Print summary table ───────────────────────────────────────
print("\n" + "="*65)
print("  FINAL RESULTS SUMMARY")
print("="*65)
print(f"  {'Model':<35} {'Accuracy':>10} {'ROC-AUC':>10} {'PR-AUC':>10}")
print(f"  {'-'*60}")
for res in all_results:
    print(f"  {res['name']:<35} {res['acc']:>10.4f} {res['roc']:>10.4f} {res['pr']:>10.4f}")
print("="*65)



best = max(all_results, key=lambda x: x['acc'])
print(f"\n  Best model: {best['name']}  →  {best['acc']*100:.2f}% accuracy")